# Step 12d — Synthesize positive instances (LLM injection, FROZEN)

> The **one non-deterministic step**. An LLM injects `UnsafeDeployOnFork` into each 12c negative's workflow YAML, producing synthetic **positives**. It runs in the sibling **`smell-synthesizer`** repo; its output is **frozen** here as a versioned input. This notebook only **verifies** the frozen artifact — it does **not** call the model.

**Paper section:** Step 3 — Ground-truth construction / Synthetic positives.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/12c_synthesis_targets_UnsafeDeployOnFork_absent.json` | JSON (50 negatives) | Step 12c |

## Outputs (frozen)
| Path | Format | Consumer |
|---|---|---|
| `data/12d_synth_UnsafeDeployOnFork_present.json` | JSON (49 synthetic positives, `##synth` keys) | Step 13 (merge) |

## Why frozen (reproducibility)
The injection uses `claude-opus-4-8` with **no seed and no temperature knob**, so the YAML is not byte-reproducible. Re-running samples different content. The frozen `12d` **is** the study artifact; everything downstream (Step 13 merge, Step 12e kappa) is deterministic from it. This notebook asserts the frozen file's SHA and prints provenance.


In [1]:
# --- Notebook bootstrap ---
from dataset_utils.notebook_bootstrap import setup

setup()

import hashlib
import json
import sys
from pathlib import Path

# Guarded __file__ hack: make dataset_utils importable when run as a plain script
# without an installed wheel. Harmless in Jupyter (__file__ undefined). Do NOT remove.
try:
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
except NameError:
    pass

# Step 12d is the ONLY non-deterministic step: an LLM injects UnsafeDeployOnFork
# into each 12c negative's workflow YAML. It runs in the smell-synthesizer repo
# (`uv run smell-inject`); the output is FROZEN here. This notebook VERIFIES the
# frozen artifact and does NOT call the model.
FROZEN = Path("data/12d_synth_UnsafeDeployOnFork_present.json")
EXPECTED_SHA = "83156e3ce35c334068a871464d1f1832fea8e725e21d0cd2ede805525cac5e2b"

data = FROZEN.read_bytes()
sha = hashlib.sha256(data).hexdigest()
n = len(json.loads(data))
status = "OK" if sha == EXPECTED_SHA else "MISMATCH"
print(f"Step 12d — frozen synthetic positives: {FROZEN}")
print(f"  records : {n}")
print(f"  sha256  : {sha}  [{status}]")
assert sha == EXPECTED_SHA, (
    "12d frozen artifact changed — if this was intentional, re-freeze and update "
    "EXPECTED_SHA + the provenance block below."
)

print("""
Provenance (how 12d was generated — do NOT re-run for replication)
------------------------------------------------------------------
  repo   : smell-synthesizer @ commit 598c3b7
  tool   : uv run smell-inject
  input  : data/12c_synthesis_targets_UnsafeDeployOnFork_absent.json (50 negatives)
  model  : claude-opus-4-8 (Anthropic); max_tokens=8192; effort=medium; no seed/temperature
  output : 49 synthetic positives (##synth keys; 1 of 50 skipped by the model)
  review : data/12e_sinth_manual_evaluation.csv (A2/A3) -> 45 approved / 4 rejected
Non-deterministic: re-running smell-inject samples different YAML. The frozen 12d
above IS the study artifact; Step 13 (merge) and Step 12e (kappa) are deterministic from it.
""")


Step 12d — frozen synthetic positives: data/12d_synth_UnsafeDeployOnFork_present.json
  records : 49
  sha256  : 83156e3ce35c334068a871464d1f1832fea8e725e21d0cd2ede805525cac5e2b  [OK]

Provenance (how 12d was generated — do NOT re-run for replication)
------------------------------------------------------------------
  repo   : smell-synthesizer @ commit 598c3b7
  tool   : uv run smell-inject
  input  : data/12c_synthesis_targets_UnsafeDeployOnFork_absent.json (50 negatives)
  model  : claude-opus-4-8 (Anthropic); max_tokens=8192; effort=medium; no seed/temperature
  output : 49 synthetic positives (##synth keys; 1 of 50 skipped by the model)
  review : data/12e_sinth_manual_evaluation.csv (A2/A3) -> 45 approved / 4 rejected
Non-deterministic: re-running smell-inject samples different YAML. The frozen 12d
above IS the study artifact; Step 13 (merge) and Step 12e (kappa) are deterministic from it.



---

**Next step:** [`12e_recalc_kappa.ipynb`](./12e_recalc_kappa.ipynb) (recompute kappa with the validated synthetic evaluations) and [`13_final_ground_truth.ipynb`](./13_final_ground_truth.ipynb) (merge into the final ground truth).